<a href="https://colab.research.google.com/github/PemaDT/CompStat-3.0-Urban_Safety-Framework/blob/main/notebooks/%20M2_Data_Ingestion_and_Processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Run this once per session
!pip install sodapy pandas pyarrow fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 41.2 MB/s eta 0:00:00


In [ ]:
# Setup & Authenticate
import pandas as pd
import os
import gc
import glob
from sodapy import Socrata
from google.colab import userdata, drive

# Mount Drive first to ensure paths exist
drive.mount('/content/drive')

# Securely grab token from Colab Secrets
APP_TOKEN = userdata.get('NYC_TOKEN')
client = Socrata("data.cityofnewyork.us", APP_TOKEN, timeout=120)

# Create project folders
RAW_PATH = '/content/drive/MyDrive/CompStat3_Project/data/raw/'
CHUNK_PATH = os.path.join(RAW_PATH, 'chunks_311/')
os.makedirs(CHUNK_PATH, exist_ok=True)

print("✅ Authentication Successful & Folders Ready!")

Mounted at /content/drive
✅ Authentication Successful & Folders Ready!


In [ ]:
#Pull NYPD Arrest Data (2021-2025)
def pull_arrests(dataset_id, label):
    print(f"⏳ Pulling {label}...")
    results = []
    offset = 0
    while True:
        chunk = client.get(dataset_id, where="arrest_date >= '2021-01-01T00:00:00'",
                           select="arrest_date, arrest_precinct, ofns_desc, pd_desc, latitude, longitude",
                           limit=50000, offset=offset)
        if not chunk: break
        results.extend(chunk)
        offset += 50000
        print(f"   ...fetched {offset:,} rows")
    return pd.DataFrame.from_records(results)

# Pull and Combine
df_hist = pull_arrests("8h9b-rp9u", "Historic Arrests")
df_ytd = pull_arrests("uip8-fykc", "YTD Arrests")
df_arrests = pd.concat([df_hist, df_ytd]).drop_duplicates()

# Clean dates and save
df_arrests['arrest_date'] = pd.to_datetime(df_arrests['arrest_date'])
df_arrests = df_arrests[df_arrests['arrest_date'] >= '2021-01-01']
df_arrests.to_parquet(f'{RAW_PATH}arrests_2021_2025.parquet', index=False)

del df_hist, df_ytd
gc.collect()
print(f"✅ Saved Arrests: {len(df_arrests):,} rows")

⏳ Pulling Historic Arrests...
   ...fetched 50,000 rows
   ...fetched 100,000 rows
   ...fetched 150,000 rows
   ...fetched 200,000 rows
   ...fetched 250,000 rows
   ...fetched 300,000 rows
   ...fetched 350,000 rows
   ...fetched 400,000 rows
   ...fetched 450,000 rows
   ...fetched 500,000 rows
   ...fetched 550,000 rows
   ...fetched 600,000 rows
   ...fetched 650,000 rows
   ...fetched 700,000 rows
   ...fetched 750,000 rows
   ...fetched 800,000 rows
   ...fetched 850,000 rows
⏳ Pulling YTD Arrests...
   ...fetched 50,000 rows
   ...fetched 100,000 rows
   ...fetched 150,000 rows
   ...fetched 200,000 rows
   ...fetched 250,000 rows
   ...fetched 300,000 rows
✅ Saved Arrests: 932,820 rows


In [ ]:
#Pulling 311 data (Memory-Safe Chunking)
# Updated Cell 4: RESUME Logic
import glob

# 1. Check how many rows we already have
existing_files = glob.glob(f'{CHUNK_PATH}chunk_*.parquet')
if existing_files:
    # Calculate next offset: (Number of files) * (Chunk Size)
    # Since we saved in 50,000 increments:
    offset = len(existing_files) * 50000
    file_idx = len(existing_files)
    print(f"🔄 Resuming from row {offset:,} (Chunk {file_idx})...")
else:
    offset = 0
    file_idx = 0
    print("⏳ Starting fresh pull...")

print("⏳ Pulling remaining 311 data...")

while True:
    try:
        chunk = client.get("erm2-nwe9",
                           where=f"created_date >= '2021-01-01T00:00:00' AND complaint_type IN ({QOL_FILTER})",
                           select="created_date, complaint_type, descriptor, latitude, longitude",
                           limit=50000, offset=offset)
        if not chunk: break

        df_chunk = pd.DataFrame.from_records(chunk)
        df_chunk.to_parquet(f'{CHUNK_PATH}chunk_{file_idx:04d}.parquet', index=False)

        offset += 50000
        file_idx += 1
        del df_chunk
        gc.collect()
        if file_idx % 5 == 0: print(f"   ✔ Processed {offset:,} rows...")

    except Exception as e:
        print(f"\n❌ Timed out again at {offset:,}. Save and run this cell again to resume!")
        break

print(f"✅ Acquisition status: {offset:,} rows total in Drive.")

🔄 Resuming from row 6,850,000 (Chunk 137)...
⏳ Pulling remaining 311 data...
   ✔ Processed 7,000,000 rows...
   ✔ Processed 7,250,000 rows...
   ✔ Processed 7,500,000 rows...
✅ Acquisition status: 7,500,000 rows total in Drive.


In [ ]:
#Final Merge( Memory-Optimized)
# 1. Find all chunks you just pulled
chunk_files = sorted(glob.glob(f'{CHUNK_PATH}chunk_*.parquet'))
print(f"⏳ Found {len(chunk_files)} chunk files in your Drive.")

if len(chunk_files) == 0:
    print("❌ Error: No chunks found! Check your folder path.")
else:
    print("⏳ Combining chunks into one final file... this may take a few minutes.")

    # We read all chunks into a list and concat them at once
    # This is faster than adding them one by one
    df_311 = pd.concat([pd.read_parquet(f) for f in chunk_files], ignore_index=True)

    # Ensure dates are in the correct format
    df_311['created_date'] = pd.to_datetime(df_311['created_date'])

    # Save the final consolidated file to your 'raw' folder
    final_save_path = os.path.join(RAW_PATH, '311_2021_2025.parquet')
    df_311.to_parquet(final_save_path, index=False)

    print(f"✅ Final 311 Dataset Created: {len(df_311):,} rows")
    print(f"✅ Saved to: {final_save_path}")

    # Clear memory immediately after saving
    # We keep df_311 in memory for the final sanity check in Cell 7
    gc.collect()

⏳ Found 150 chunk files in your Drive.
⏳ Combining chunks into one final file... this may take a few minutes.
✅ Final 311 Dataset Created: 7,466,076 rows
✅ Saved to: /content/drive/MyDrive/CompStat3_Project/data/raw/311_2021_2025.parquet


In [ ]:
# Run this to delete the individual chunks and save space in your Google Drive
# ONLY run this if Cell 5 says "Final 311 Dataset Created"
confirm = input("Type 'yes' to delete individual chunks and save Drive space: ")
if confirm.lower() == 'yes':
    for f in chunk_files:
        os.remove(f)
    print("🧹 Chunks deleted. Your Drive is clean!")
else:
    print("Skipped cleanup. Chunks remain in folder.")

Type 'yes' to delete individual chunks and save Drive space: yes
🧹 Chunks deleted. Your Drive is clean!


In [ ]:
#The milestone 2 Sanity Check
print("\n" + "="*50)
print("📊 FINAL MILESTONE 2 PROJECT SUMMARY")
print("="*50)

# Check Arrests (Loaded from the file we saved in Cell 3)
df_arr_check = pd.read_parquet(os.path.join(RAW_PATH, 'arrests_2021_2025.parquet'))

print(f"Total NYPD Arrest Records:   {len(df_arr_check):,}")
print(f"Total 311 QOL Records:       {len(df_311):,}")
print("-" * 50)
print(f"Arrest Date Range:  {df_arr_check['arrest_date'].min().date()} to {df_arr_check['arrest_date'].max().date()}")
print(f"311 Date Range:     {df_311['created_date'].min().date()} to {df_311['created_date'].max().date()}")
print("-" * 50)
print("Top 5 311 Complaint Types Ingested:")
print(df_311['complaint_type'].value_counts().head(5))
print("="*50)
print("✅ Milestone 2: Data Acquisition is COMPLETE.")


📊 FINAL MILESTONE 2 PROJECT SUMMARY
Total NYPD Arrest Records:   932,820
Total 311 QOL Records:       7,466,076
--------------------------------------------------
Arrest Date Range:  2021-01-01 to 2025-12-31
311 Date Range:     2021-01-01 to 2026-04-15
--------------------------------------------------
Top 5 311 Complaint Types Ingested:
complaint_type
Illegal Parking            2456793
Noise - Residential        1983378
Blocked Driveway            877916
Noise - Street/Sidewalk     846557
Abandoned Vehicle           333116
Name: count, dtype: int64
✅ Milestone 2: Data Acquisition is COMPLETE.
